# The Price is Right

## 第 8 周日程安排

第 1 天：Modal.com 与 SpecialistAgent  
第 2 天：RAG、FrontierAgent、Ensemble Agent  
第 3 天：ScannerAgent、MessengerAgent  
第 4 天：AutonomousPlannerAgent 与 DealAgentFramework  
第 5 天：The Price Is Right 终章


今天我们会搭建拼图的另一块：一个通过订阅 RSS feeds 来寻找有潜力优惠的 ScanningAgent。

In [ ]:
# 导入：爬取 deals、结构化输出 DealSelection、HTTP（Pushover）
# 今天做 Scanner Agent：从网上抓优惠并用 LLM 精选

import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [ ]:
# 爬取最新优惠列表（可能需一些时间）

deals = ScrapedDeal.fetch(show_progress=True)

In [ ]:
# 看看抓到了多少条

len(deals)

In [ ]:
# 查看其中一条 deal 的描述文本

deals[10].describe()

### 我们将让 GPT-5-mini 总结优惠并识别其价格

In [ ]:
# 系统/用户提示词模板：要求模型挑出 5 条描述最详细且价格清晰的优惠
# 注意：不要改动下面的英文 prompt 字符串内容

SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [ ]:
# 根据爬取的 deals 生成合适的 user prompt

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [ ]:
# 为我们刚爬取的 deals 创建 user prompt，并看看它开头是什么样

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

In [ ]:
# 结构化解析：让模型直接返回 DealSelection（含产品描述、价格、URL）

response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

In [ ]:
# 打印模型精选出的 5 条优惠

for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


In [ ]:
# 打开日志，便于观察 ScannerAgent 内部流程

root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
# ScannerAgent：把「爬取 + LLM 精选」封装成可复用 Agent

from agents.scanner_agent import ScannerAgent

In [ ]:
# 运行完整扫描流程

agent = ScannerAgent()
result = agent.scan()

In [ ]:
# 查看扫描结果对象

result

### 介绍 Pushover

Pushover 是一个把推送通知发到手机的便捷工具。

安装和设置都超级简单！

只需访问 https://pushover.net/，点击右上角的「Login or Signup」免费注册账号，并创建你的 API keys。

注册后，在主页点击「Create an Application/API Token」，随便起个名字（比如 AIEngineer），然后点击 Create Application。

然后在你的 `.env` 文件中加入两行：

PUSHOVER_USER=_填入 Pushover 主页右上角那个 key，通常以 u 开头_  
PUSHOVER_TOKEN=_填入点击进入你新建的应用（比如叫 Agents 或任意名称）后看到的那个 key，通常以 a 开头_

记得保存 `.env` 文件，保存后运行 `load_dotenv(override=True)` 以设置环境变量。

最后，点击「Add Phone, Tablet or Desktop」安装到手机上。

In [ ]:
# 重新加载 .env，准备读 Pushover 推送凭证

load_dotenv(override=True)

In [ ]:
# Pushover：把「超值优惠」推送到手机的通知服务

pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
# 检查凭证是否已配置（只打印首字符，避免泄露完整密钥）

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
# 封装一次推送请求

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
# 发一条测试通知到手机

push("MASSIVE DEAL!!")

In [ ]:
# MessagingAgent：正式封装推送逻辑（供框架其他 Agent 调用）

from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
# notify：按「描述、成交价、估值、链接」发送一条优惠提醒

agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")